In [31]:
!pip install gradio PyPDF2 python-docx matplotlib reportlab

TEXT EXPLANATION
This cell installs all required libraries needed for the project.
Libraries provide ready-made tools for reading resumes,
analyzing text, generating charts, and building the web interface.

In [32]:
# INSTALL
import gradio as gr
import PyPDF2
import docx
import re
import matplotlib.pyplot as plt
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet



# TEXT EXPLANATION
# This cell imports the installed libraries into Python.
# Importing allows us to use their features in the program.

In [33]:
# GLOBAL VARIABLES
resume_text=""
detected_bullets=[]

skills_db=[
"python","machine learning","data science","deep learning",
"sql","tableau","power bi","excel","tensorflow","nlp",
"pandas","numpy","java","c++","javascript","ai"
]


# TEXT EXPLANATION
# This cell stores important variables used across the project.
# skills_db used for resume analysis.

In [34]:
# ---------------- UPLOAD ----------------
def upload_resume(file):

    global resume_text
    resume_text=""

    if file.name.endswith(".pdf"):
        pdf=PyPDF2.PdfReader(file)
        for page in pdf.pages:
            text=page.extract_text()
            if text:
                resume_text+=text

    elif file.name.endswith(".docx"):
        doc=docx.Document(file)
        for p in doc.paragraphs:
            resume_text+=p.text

    return "Resume uploaded successfully!"



# TEXT EXPLANATION
# This function allows the user to upload a resume file.
# The program reads the file and extracts all text from it.
# This text will later be used for analysis.

In [35]:


# ---------------- SCORE ----------------
def resume_score():

    if resume_text=="":
        return None,"Upload resume first"

    score=60

    if "project" in resume_text.lower():
        score+=10
    if "experience" in resume_text.lower():
        score+=10
    if "skill" in resume_text.lower():
        score+=10

    fig,ax=plt.subplots()

    ax.pie([score,100-score],startangle=90,
           colors=["#2ecc71","#eeeeee"],
           wedgeprops={"width":0.35})

    ax.text(0,0,f"{score}/100",ha="center",va="center")

    return fig,f"Resume Score: {score}/100"



# TEXT EXPLANATION
# This function evaluates the overall strength of the resume.
# It checks if important sections like projects, skills,
# and experience are present and generates a score.

In [36]:
# ---------------- SKILLS ----------------
def detect_skills():

    if resume_text == "":
        return None, "Upload resume first"

    # detect skills (same as your logic)
    found = [s.upper() for s in skills_db if s in resume_text.lower()]

    # if no skills found
    if len(found) == 0:
        return None, "No skills detected in resume"

    # create graph (slight fix: size + spacing)
    fig, ax = plt.subplots(figsize=(7,5))

    ax.barh(found, [80]*len(found))

    ax.set_title("Detected Skills")

    plt.tight_layout()  # FIX: prevents text cutting

    return fig, f"Detected Skills: {found}"


# TEXT EXPLANATION
# This function scans the resume and detects technical skills.
# It compares the resume text with a predefined list of skills.

In [37]:
def ats_checker(job_desc):

    if resume_text=="":
        return "Upload resume first"

    keywords=re.findall(r'\b\w+\b',job_desc.lower())

    matched=[k for k in keywords if k in resume_text.lower()]
    missing=[k for k in keywords if k not in resume_text.lower()]

    score=int((len(matched)/len(keywords))*100) if keywords else 0

    return f"""
ATS Score: {score}%

Matched:
{matched}

Missing:
{missing}
"""


# TEXT EXPLANATION
# ATS stands for Applicant Tracking System.
# Many companies use ATS software to automatically filter resumes.
# This function checks if important keywords are present.

In [38]:

# ---------------- BULLETS ----------------
def extract_bullets():

    global detected_bullets

    if resume_text=="":
        return "Upload resume first"

    detected_bullets=[]

    parts=re.split(r'[\n•\-●▪.]',resume_text)

    for line in parts:
        line=line.strip()
        if len(line)>25:
            detected_bullets.append("• "+line)

    if not detected_bullets:
        detected_bullets=["• Resume content found but no structured bullets detected"]

    return "\n".join(detected_bullets[:10])



# TEXT EXPLANATION
# This function automatically detects bullet points from the resume.
# Bullet points usually describe achievements, tasks, or skills.

In [39]:
# ---------------- BULLET AI IMPROVER ----------------
def improve_bullet():

    if not detected_bullets:
        return "No bullets detected"

    improved=[]

    for b in detected_bullets:

        text=b.lower()

        if "python" in text:
            new="Developed scalable and high-performance applications using Python with strong optimization techniques."

        elif "machine learning" in text:
            new="Designed and implemented machine learning models to solve complex real-world problems."

        elif "data" in text:
            new="Performed in-depth data analysis to extract actionable insights and improve decision-making."

        elif "project" in text:
            new="Successfully executed end-to-end projects demonstrating technical expertise and problem-solving ability."

        else:
            new="Applied strong analytical and technical skills to deliver efficient and scalable solutions."

        improved.append("• "+new)

    return "\n\n".join(improved)


# TEXT EXPLANATION
# This function improves the bullet points extracted from the resume
# by rewriting them with stronger professional wording.

In [40]:

# ---------------- AI ANALYZER ----------------
def ai_resume_analyzer():

    if resume_text=="":
        return "Upload resume first"

    feedback=[]

    if len(resume_text)<800:
        feedback.append("Increase content with detailed project explanations.")

    if "project" not in resume_text.lower():
        feedback.append("Add a strong project section.")

    if "experience" not in resume_text.lower():
        feedback.append("Include experience or internship.")

    if "achievement" not in resume_text.lower():
        feedback.append("Add measurable achievements.")

    if not feedback:
        feedback.append("Your resume is strong and well-structured.")

    return "\n\n".join(feedback)



# TEXT EXPLANATION
# This function analyzes the resume and provides suggestions
# for improving the overall quality.

In [41]:
# ---------------- RESUME REWRITER ----------------
def rewrite_resume():

    if resume_text=="":
        return "Upload resume first",None

    skills=[s.upper() for s in skills_db if s in resume_text.lower()]

    summary="Highly skilled and motivated professional with strong technical expertise and problem-solving abilities."

    lines=re.split(r'[.\n]',resume_text)

    exp=[]

    for l in lines:
        if len(l.strip())>25:
            exp.append("• "+l.strip())

    final=f"""
PROFESSIONAL SUMMARY:
{summary}

TECHNICAL SKILLS:
{", ".join(skills) if skills else "General Skills"}

EXPERIENCE:
"""+"\n".join(exp[:6])

    file_path="resume.pdf"

    styles=getSampleStyleSheet()
    doc=SimpleDocTemplate(file_path)
    doc.build([Paragraph(final,styles["Normal"])])

    return final,file_path



# TEXT EXPLANATION
# This function rewrites the entire resume using better wording
# and generates a downloadable improved resume.

In [42]:
# ---------------- CSS ----------------
css="""
.card {
padding:20px;
border-radius:15px;
color:white;
font-size:18px;
text-align:center;
font-weight:bold;
}
textarea {height:250px !important;}
"""



# TEXT EXPLANATION
# This section defines the design and colors of the interface.

In [43]:
# ---------------- UI ----------------
with gr.Blocks(css=css) as app:

    with gr.Tab("Home"):
        gr.Markdown("## Generative AI Resume Analyzer & Improver")

        with gr.Row():
            gr.Markdown("<div class='card' style='background:#ff4fa3'>Resume Score</div>")
            gr.Markdown("<div class='card' style='background:#3498db'>Skill Detection</div>")

        with gr.Row():
            gr.Markdown("<div class='card' style='background:#2ecc71'>ATS Checker</div>")
            gr.Markdown("<div class='card' style='background:#ff8c42'>Bullet Improver</div>")

        with gr.Row():
            gr.Markdown("<div class='card' style='background:#e63946'>Resume Analyzer</div>")
            gr.Markdown("<div class='card' style='background:#9b59b6'>Resume Rewriter</div>")

    with gr.Tab("Upload Resume"):
        file_input=gr.File()
        upload_btn=gr.Button("Upload Resume")
        upload_output=gr.Textbox()
        upload_btn.click(upload_resume,file_input,upload_output)

    with gr.Tab("Resume Score"):
        btn=gr.Button("Analyze")
        plot=gr.Plot()
        text=gr.Textbox()
        btn.click(resume_score,outputs=[plot,text])

    with gr.Tab("Skill Detection"):
        btn=gr.Button("Detect")
        plot=gr.Plot()
        text=gr.Textbox()
        btn.click(detect_skills,outputs=[plot,text])

    with gr.Tab("ATS Checker"):
        inp=gr.Textbox(label="Enter Job Description",lines=5)
        btn=gr.Button("Check")
        out=gr.Textbox(lines=10)
        btn.click(ats_checker,inp,out)

    with gr.Tab("Bullet Improver"):
        box=gr.Textbox(lines=12)
        d=gr.Button("Detect Bullets")
        i=gr.Button("Improve Bullets")
        out=gr.Textbox(lines=12)
        d.click(extract_bullets,outputs=box)
        i.click(improve_bullet,outputs=out)

    with gr.Tab("AI Resume Analyzer"):
        btn=gr.Button("Analyze")
        out=gr.Textbox(lines=12)
        btn.click(ai_resume_analyzer,outputs=out)

    with gr.Tab("Resume Rewriter"):
        btn=gr.Button("Rewrite Resume")
        txt=gr.Textbox(lines=15)
        file=gr.File()
        btn.click(rewrite_resume,outputs=[txt,file])

# ---------------- LAUNCH ----------------

/tmp/ipykernel_5426/2724829532.py:2: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css) as app:


# TEXT EXPLANATION
# This cell creates the graphical interface of the application.

In [44]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5365f71e5afb302c35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# TEXT EXPLANATION
# This command launches the application and generates a public link
# so the user can open the interface in a browser.